# Agent Test Notebook

Test each orchestrator agent and tool from `backend/agent/orchestrator.py` individually: **intent classifier**, **planner**, **evaluator**, **executor tools**, **validator**, and **response generator**. Use the cells below to run with custom input and inspect output.

**Prerequisites:** Run from project root (or open from repo root so cwd is correct). Set `LLM_PROVIDER` and related env (e.g. `OLLAMA_MODEL`, `OPENAI_API_KEY`) as needed. For tools that use the DB (order_lookup, return_initiate), ensure Postgres is configured. Optional: `pip install nest_asyncio` if you see "event loop is already running" when calling async agents.

In [1]:
# Setup: add project root so we can import backend
import sys
import os
import asyncio
import json

# Project root: cwd when opened from project root, or parent when cwd is notebooks/
PROJECT_ROOT = os.path.abspath(".")
if os.path.basename(PROJECT_ROOT) == "notebooks" or not os.path.isdir(os.path.join(PROJECT_ROOT, "backend")):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Allow running async in Jupyter if event loop is already running
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

from backend.agent.state import AgentState
from backend.agent import orchestrator
from backend.config import get_llm
from backend.observability.langsmith_tracer import build_stage_run_config
from langchain_core.messages import HumanMessage, SystemMessage

In [2]:
def make_state(
    user_message: str,
    session_id: str = "test-session",
    user_id: str | None = None,
    request_id: str | None = None,
) -> AgentState:
    """Build minimal AgentState for testing with a single user message."""
    return {
        "messages": [{"role": "user", "content": user_message}],
        "session_id": session_id,
        "user_id": user_id,
        "request_id": request_id or "test-req",
    }


def run_async(coro):
    """Run an async function in the notebook. Uses nest_asyncio if loop is already running."""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio
            nest_asyncio.apply()
    except Exception:
        pass
    return asyncio.get_event_loop().run_until_complete(coro)

## 1. Intent Classifier

Classifies user message into intent, sentiment score, and whether the user asked for a human. Output: `intent`, `classifier_sentiment_score`, `user_requested_human`.

In [ ]:
# Test input — change this to try different phrases
user_input = "Where is my order ORD-12345?"
state = make_state(user_input)

out = run_async(orchestrator.classify_intent_only(state))
print("Input:", user_input)
print("Intent:", out.get("intent"))
print("Sentiment (0–1):", out.get("classifier_sentiment_score"))
print("User requested human:", out.get("user_requested_human"))
print("Raw classifier (workflow_state):", out.get("workflow_state", {}).get("classifier"))

## 2. Tools (Executor)

Test each tool used by the executor: **faq_search**, **order_lookup**, **return_initiate**. Adjust parameters and inspect returned payloads.

In [ ]:
from backend.tools.faq_search import faq_search_tool

result = run_async(faq_search_tool(query="shipping policy", category=None, top_k=3))
print(json.dumps(result, indent=2, default=str))

In [ ]:
from backend.tools.order_lookup import order_lookup_tool

# Provide order_number and/or user_id (depends on your DB data)
result = run_async(order_lookup_tool(order_number="ORD-12345", user_id=None))
print(json.dumps(result, indent=2, default=str))

In [ ]:
from backend.tools.return_initiate import return_initiate_tool

# Requires a delivered order within return window in DB
result = run_async(return_initiate_tool(order_number="ORD-12345", user_id=None))
print(json.dumps(result, indent=2, default=str))

## 3. Planner (standalone)

Call the planner LLM with intent + user request + optional feedback. Input: intent, user_request, cycle_count, previous_feedback. Output: plan JSON (plan_id, tasks, metadata).

In [ ]:
user_request = "I want to know the status of my order ORD-999."
intent = "order_status"
cycle_count = 1
previous_feedback = ""

state = make_state(user_request)
planner_input = json.dumps(
    {
        "intent": intent,
        "user_request": user_request,
        "cycle_count": cycle_count,
        "previous_feedback": previous_feedback,
    },
    ensure_ascii=False,
)

llm = get_llm(role="planner")
resp = run_async(
    llm.ainvoke(
        [
            SystemMessage(content=orchestrator.SYSTEM_PLANNER_SCHEMA),
            HumanMessage(content=planner_input),
        ],
        config=build_stage_run_config(state, "agent.planner", cycle_count=cycle_count),
    )
)
raw = (resp.content or "").strip()
print("Planner raw output:")
print(raw)
parsed = orchestrator._extract_json(raw)
if parsed:
    print("\nParsed plan:")
    print(json.dumps(parsed, indent=2, default=str))

## 4. Plan Evaluator (standalone)

Validate a plan against user_request. Input: user_request, plan. Output: plan_valid, feedback.

In [ ]:
user_request = "Where is my order?"
plan = {
    "plan_id": "p1",
    "tasks": [
        {
            "task_id": "t1",
            "action": "order_lookup",
            "params": {"order_number": None, "user_id": "user-1"},
            "depends_on": [],
        }
    ],
    "metadata": {"strategy": "lookup by user", "user_request": user_request},
}

state = make_state(user_request)
evaluator_input = json.dumps({"user_request": user_request, "plan": plan}, ensure_ascii=False)

llm = get_llm(role="small")
resp = run_async(
    llm.ainvoke(
        [
            SystemMessage(content=orchestrator.SYSTEM_PLAN_EVALUATOR),
            HumanMessage(content=evaluator_input),
        ],
        config=build_stage_run_config(state, "agent.evaluator"),
    )
)
raw = (resp.content or "").strip()
print("Evaluator raw:", raw)
parsed = orchestrator._extract_json(raw)
if parsed:
    print("Valid:", parsed.get("plan_valid"), "|", "Feedback:", parsed.get("feedback"))

## 5. Validator (standalone)

Check if plan + execution result achieved the user request. Input: plan, result. Output: achieved, feedback, sentiment_score.

In [ ]:
plan = {
    "plan_id": "p1",
    "tasks": [{"task_id": "t1", "action": "order_lookup", "params": {}, "depends_on": []}],
    "metadata": {"user_request": "Where is my order?"},
}
result = {
    "tasks": [
        {
            "task_id": "t1",
            "action": "order_lookup",
            "success": True,
            "result": {"orders": [{"order_number": "ORD-1", "status": "shipped"}]},
        }
    ],
    "by_task_id": {},
}

state = make_state("Where is my order?")
validator_input = json.dumps({"plan": plan, "result": result}, ensure_ascii=False)

llm = get_llm(role="small")
resp = run_async(
    llm.ainvoke(
        [
            SystemMessage(content=orchestrator.SYSTEM_VALIDATOR),
            HumanMessage(content=validator_input),
        ],
        config=build_stage_run_config(state, "agent.validator"),
    )
)
raw = (resp.content or "").strip()
print("Validator raw:", raw)
parsed = orchestrator._extract_json(raw)
if parsed:
    print("Achieved:", parsed.get("achieved"), "|", "Feedback:", parsed.get("feedback"), "|", "Sentiment:", parsed.get("sentiment_score"))

## 6. Response Generator (standalone)

Generate final user-facing text from plan + result. Input: plan, result. Output: plain-text response.

In [ ]:
plan = {
    "plan_id": "p1",
    "metadata": {"user_request": "Where is my order?"},
}
result = {
    "tasks": [
        {
            "task_id": "t1",
            "action": "order_lookup",
            "success": True,
            "result": {"orders": [{"order_number": "ORD-1", "status": "shipped"}]},
        }
    ],
}

state = make_state("Where is my order?")
gen_input = json.dumps({"plan": plan, "result": result}, ensure_ascii=False)

llm = get_llm(role="small")
resp = run_async(
    llm.ainvoke(
        [
            SystemMessage(content=orchestrator.SYSTEM_RESPONSE_GENERATOR),
            HumanMessage(content=gen_input),
        ],
        config=build_stage_run_config(state, "agent.response_generator"),
    )
)
print("Generated response:")
print(resp.content or "")

## 7. Full pipeline

Run the full orchestrated agent: intent → planner → evaluator → executor → validator (loop) → response (or escalation). Inspect `final_response`, `plan`, `execution_results`, `should_escalate`, etc.

In [ ]:
user_message = "What is your return policy?"
state = make_state(user_message)

out = run_async(orchestrator.run_orchestrated_agent(state))

print("Input:", user_message)
print("Intent:", out.get("intent"))
print("Should escalate:", out.get("should_escalate"))
print("Final response:", out.get("final_response"))
print("\nPlan:")
print(json.dumps(out.get("plan") or {}, indent=2, default=str))
print("\nExecution results:")
print(json.dumps(out.get("execution_results") or {}, indent=2, default=str))